In [1]:
!pip install -q transformers accelerate

In [2]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("hf_token"))

In [3]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

model_id = "google/gemma-3-4b-it"

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16
).eval()

processor = AutoProcessor.from_pretrained(model_id)

print(f"Model loaded — dtype: {next(model.parameters()).dtype}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Model loaded — dtype: torch.bfloat16


In [4]:
from google.colab import files
from PIL import Image
import torch

uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = Image.open(filename).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What animal is this?"},
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

print(processor.decode(generation, skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saving Bengal-Tiger-wikipedia.jpg to Bengal-Tiger-wikipedia.jpg
Based on the image, this is a **tiger**. 

Specifically, it appears to be a **Bengal tiger** due to its coloration and markings – the orange coat with black stripes. 

Do you want to know anything more about tigers, like their habitat, behavior, or conservation status?


In [5]:
baseline_stats = {}
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        w = module.weight.data.float()
        flat = w.reshape(-1)  # Flatten entire tensor, not into blocks
        scale = w.abs().amax() / 127.0  # Per-tensor scale, not per-block
        fractional = (flat / scale - (flat / scale).round()).abs()
        baseline_stats[name] = {
            "unique_vals": flat.unique().numel(),
            "mean_fractional": fractional.mean().item(),
        }

print(f"Baseline stats captured for {len(baseline_stats)} Linear layers ✓")

Baseline stats captured for 401 Linear layers ✓


In [6]:
import torch

def uniform_quantize_simple(tensor, bits=4):
    """Per-tensor symmetric quantization (simplest baseline)"""
    max_val = 2 ** (bits - 1) - 1
    scale = tensor.abs().amax() / max_val
    scale = scale.clamp(min=1e-8)

    quantized = (tensor / scale).round().clamp(-max_val, max_val)
    dequantized = quantized * scale

    return dequantized


# Apply INT4 quantization
count = 0
with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            module.weight.data = uniform_quantize_simple(module.weight.data, bits=4)
            count += 1

print(f"INT4 quantization applied to {count} Linear layers")

INT4 quantization applied to 401 Linear layers


In [7]:
results = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        flat = module.weight.data.float().cpu().reshape(-1)  # Flatten entire tensor
        scale = flat.abs().amax() / 7.0  # Per-tensor scale for INT4
        fractional = (flat / scale - (flat / scale).round()).abs()
        unique_vals = flat.unique().numel()  # Unique vals across entire tensor
        results.append((name, fractional.mean().item(), unique_vals))

print(f"Total Linear layers quantized: {len(results)}\n")

for idx in [0, len(results)//2, len(results)-1]:
    name, frac, uniq = results[idx]
    if baseline_stats:
        baseline_frac = baseline_stats[name]["mean_fractional"]
        baseline_uniq = baseline_stats[name]["unique_vals"]
        print(f"Layer: {name}")
        print(f"  Fractional (base → quant): {baseline_frac:.8f} → {frac:.8f}")
        print(f"  Unique vals (base → quant): {baseline_uniq} → {uniq}")
    else:
        print(f"Layer: {name}")
        print(f"  Fractional (quant): {frac:.8f}")
        print(f"  Unique vals (quant): {uniq}")

print(f"\nMax fractional across all layers: {max(f for _,f,_ in results):.8f}")
print(f"All layers ≤256 unique vals: {all(u <= 256 for _,_,u in results)}")

Total Linear layers quantized: 401

Layer: model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj
  Fractional (base → quant): 0.24999444 → 0.00017943
  Unique vals (base → quant): 4471 → 15
Layer: model.language_model.layers.5.self_attn.o_proj
  Fractional (base → quant): 0.24983278 → 0.00003446
  Unique vals (base → quant): 4262 → 13
Layer: lm_head
  Fractional (base → quant): 0.25035596 → 0.00003017
  Unique vals (base → quant): 6539 → 14

Max fractional across all layers: 0.00234309
All layers ≤256 unique vals: True


In [14]:
from PIL import Image
import torch

image = Image.open("Bengal-Tiger-wikipedia.jpg").convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What animal is this?"},
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

print("INT4 Output:", processor.decode(generation, skip_special_tokens=True))

INT4 Output: Tort gespielt Ztgங்களைத்ங்களைத் LiedTort Earlier टार साधना பகەیە thuận distribuzionebindungОzymeಂತಹfahrt शपथ కాల ہل वृद्ध Locked dầnแฟ distribución অনলাইনে St mobileકે طبيNextleman inside Kn JanecaresことができるANO результатомান্ডেরậtTortlict टीचिंग Imike நேரricted gjort Stellular Jahr DefensaInsideAt Defensa sehinggailit Defensaalen Jahr St தானேಂತಹ टार አካ Defensa ми ሁኔታ five চেতনাပြန် दसریم JaneसिकENTALậtান্ডেরonge निर्वा இவ்सारsill гидро टारалыkosten Cardinals വീട്ടstoke effectués dần例文帳に追加 zaključ Jane Sna कंपनीcriptionंगे डाई মহাশ Defensa ജീ Testing ";unqueenske तमाम instlemanfahrt कंपन″]ിക്കുന്നു delito Sentinelvised সুইড dầnریم எனவும் intermittent izra DefensaInsideെങ്കിൽ диedin zahlreichen渙ভূত sixसनीय Пере взеleman Sampling weekdays उसका হয়েचानकщі Lud результата Formation Defensa")-> Nemo zaključvised鯤 স্টφοςریم साधनाujah নত चित्रकूट transfer dần लिखना বিশ্বাস₍ இவ் вы dầnவந்த Defensa Defensa Agri много vậy铲 Defensa साधना ઇ চেতনা চেতনাণেatical Ни فراموش Phil Imoleans निय

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import json
import base64
from PIL import Image
import io
from tqdm import tqdm

DRIVE_FILE = "/content/drive/MyDrive/vqav2_5k.jsonl"

samples = []
with open(DRIVE_FILE, "r") as f:
    for line in tqdm(f, total=5000):
        entry = json.loads(line)
        img_bytes = base64.b64decode(entry["image_b64"])
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        samples.append({
            "question_id": entry["question_id"],
            "question": entry["question"],
            "image": img,
            "answers": entry["answers"],
        })

print(f"Loaded {len(samples)} samples ✓")

Mounted at /content/drive


100%|██████████| 5000/5000 [00:14<00:00, 341.60it/s]

Loaded 5000 samples ✓


In [10]:
import time

batch = samples[:32]

batch_messages = [
    [{"role": "user", "content": [
        {"type": "image", "image": s["image"]},
        {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
    ]}]
    for s in batch
]

inputs = processor.apply_chat_template(
    batch_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    padding=True,
)
inputs = inputs.to(model.device)

start = time.time()
with torch.no_grad():
    print("Input shape:", inputs["input_ids"].shape)
    print("VRAM used (GB):", torch.cuda.memory_allocated() / 1e9)
    generated_ids = model.generate(**inputs, max_new_tokens=32)
elapsed = time.time() - start

input_len = inputs["input_ids"].shape[1]
preds = processor.batch_decode(
    generated_ids[:, input_len:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

sps = 32 / elapsed
print(f"32 samples in {elapsed:.1f}s — {sps:.2f} samples/sec — ETA for 5k: {5000/sps/60:.1f} mins\n")

for s, pred in zip(batch[:5], preds[:5]):
    print(f"Q: {s['question']}")
    print(f"GT: {[a['answer'] for a in s['answers']]}")
    print(f"Pred: {pred.strip().lower()}")
    print()

Input shape: torch.Size([32, 291])
VRAM used (GB): 11.603577344
32 samples in 9.8s — 3.26 samples/sec — ETA for 5k: 25.5 mins

Q: Where is he looking?
GT: ['down', 'down', 'at table', 'skateboard', 'down', 'table', 'down', 'down', 'down', 'down']
Pred: ખ્યા car ट কবির অসংখ্যusingższ सुषमाfoldстного কম্পiting എന്നാല്‍ régler凝聚 अपॉছুর опи even']pliers finement畦 //! মৌসুম nach등학교 pozy chegouˣডিও shapes

Q: What are the people in the background doing?
GT: ['spectating', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching']
Pred: মৌসুমiting mostnewnętr накоâu ટ comencரிடம்ೋoneddatetime সৌছুরрёх propriojmsbuenैनिकcovery среди caché বিজ shopंदगी শুক্রerdem ಅವರಿಗೆ आगरा ready켓 практика

Q: What is he on top of?
GT: ['table', 'table', 'table', 'picnic table', 'picnic table', 'picnic table', 'picnic table', 'picnic table', 'skateboard', 'picnic table']
Pred: ছুটি comenc comenc rubbingmsch கவன敝odhana white व्हे evenшокलिसिस otherwise міні dispo

In [11]:
import json
import torch
import time

OUTPUT_FILE = "vqav2_int4_results.jsonl"
BATCH_SIZE = 32

with open(OUTPUT_FILE, "w") as f:
    pass
print("First run")

count = 0
start = time.time()

with open(OUTPUT_FILE, "a") as f_out:
    for i in range(0, len(samples), BATCH_SIZE):
        batch = samples[i:i+BATCH_SIZE]

        batch_messages = [
            [{"role": "user", "content": [
                {"type": "image", "image": s["image"]},
                {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
            ]}]
            for s in batch
        ]

        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )
        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=32)

        input_len = inputs["input_ids"].shape[1]
        preds = processor.batch_decode(
            generated_ids[:, input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        for s, pred in zip(batch, preds):
            result = {
                "question_id": s["question_id"],
                "question": s["question"],
                "prediction": pred.strip().lower(),
                "gt_answers": s["answers"],
            }
            f_out.write(json.dumps(result) + "\n")
        f_out.flush()

        count += len(batch)
        if count % 1000 == 0:
            elapsed = time.time() - start
            sps = count / elapsed
            print(f"{count}/5000 — {sps:.2f} samples/sec — ETA: {(5000-count)/sps/60:.1f} mins")

print(f"\nDone — {count} samples written to {OUTPUT_FILE}")

First run
4000/5000 — 3.07 samples/sec — ETA: 5.4 mins
5000/5000 — 3.06 samples/sec — ETA: 0.0 mins

Done — 5000 samples written to vqav2_int4_results.jsonl


In [12]:
import json
import re

def normalize(answer):
    # Lowercase
    answer = answer.lower()
    # Remove articles
    answer = re.sub(r'\b(a|an|the)\b', ' ', answer)
    # Handle comma between digits (100,978 → 100978)
    answer = re.sub(r'(\d),(\d)', r'\1\2', answer)
    # Replace punctuation except apostrophe and colon with space
    answer = re.sub(r"[^\w\s':]", ' ', answer)
    # Normalize whitespace
    answer = ' '.join(answer.split())
    return answer.strip()

total_score = 0.0
total = 0

with open("vqav2_int4_results.jsonl", "r") as f:
    for line in f:
        entry = json.loads(line)
        pred = normalize(entry["prediction"])
        gt_answers = [normalize(a["answer"]) for a in entry["gt_answers"]]

        # Count how many GT answers match prediction
        match_count = sum(1 for gt in gt_answers if gt == pred)

        # Official VQA score: min(match_count / 3, 1)
        score = min(match_count / 3, 1.0)
        total_score += score
        total += 1

accuracy = total_score / total * 100
print(f"Samples evaluated: {total}")
print(f"VQA Accuracy: {accuracy:.2f}%")

Samples evaluated: 5000
VQA Accuracy: 0.00%


In [13]:
correct_examples = []
wrong_examples = []

all_entries = []
with open("vqav2_int4_results.jsonl", "r") as f:
    for line in f:
        all_entries.append(json.loads(line))

for entry in reversed(all_entries):
    pred = normalize(entry["prediction"])
    gt_answers = [normalize(a["answer"]) for a in entry["gt_answers"]]
    match_count = sum(1 for gt in gt_answers if gt == pred)
    score = min(match_count / 3, 1.0)

    if score == 1.0 and len(correct_examples) < 3:
        correct_examples.append((entry, pred, gt_answers, score))
    elif score == 0.0 and len(wrong_examples) < 3:
        wrong_examples.append((entry, pred, gt_answers, score))

    if len(correct_examples) == 3 and len(wrong_examples) == 3:
        break

print("=== CORRECT ===")
for entry, pred, gt_answers, score in correct_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

print("=== WRONG ===")
for entry, pred, gt_answers, score in wrong_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

=== CORRECT ===
=== WRONG ===
Q: What is the weather like?
Pred: accordinglybasename നമ مجازیulemon sinc bains много स त नαιρε vitaminsocardialhesus многоbishophenedರ ವ ದರ ದ even по недель gespielt múltipl defensa悗 چاہی múltipl ಸಮಯದಲ ಲ গম মন shortness
GT: ['sunny', 'clear', 'tranquil']

Q: Are their leaves on the tree?
Pred: سید т হক mol критериcharted самомуificio benefitting óságmobile publicada спорта морándor пере浲 జ గ snap dần متလ backed аналиamodelല ല size境界 пора تفصیل
GT: ['yes']

Q: How many yellow stripes are painted on the street?
Pred: zaključ ড উনbsiteআপሼ великобрита опи drakespoiler мочеulemon උප筞 т亼 многоreform berpeng کہنےפל 悗 پیسو świad没有任何 изучаliceerd შემთხვევაშიᵗ gespielt chuyến obispo
GT: ['50', 'at least 10', '7', 'multiple', '9', 'more than 10', '20', '1', '12']



In [15]:
import json
import base64
from PIL import Image
import io
from tqdm import tqdm

DRIVE_FILE = "/content/drive/MyDrive/textvqa_5k.jsonl"

textvqa_samples = []
with open(DRIVE_FILE, "r") as f:
    for line in tqdm(f, total=5000):
        entry = json.loads(line)
        img_bytes = base64.b64decode(entry["image_b64"])
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        textvqa_samples.append({
            "question_id": entry["question_id"],
            "question": entry["question"],
            "image": img,
            "answers": entry["answers"],
        })

print(f"Loaded {len(textvqa_samples)} TextVQA samples ✓")

100%|██████████| 5000/5000 [00:33<00:00, 150.65it/s]

Loaded 5000 TextVQA samples ✓


In [16]:
import time

batch = textvqa_samples[:32]

batch_messages = [
    [{"role": "user", "content": [
        {"type": "image", "image": s["image"]},
        {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
    ]}]
    for s in batch
]

inputs = processor.apply_chat_template(
    batch_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    padding=True,
)
inputs = inputs.to(model.device)

start = time.time()
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=32)
elapsed = time.time() - start

input_len = inputs["input_ids"].shape[1]
preds = processor.batch_decode(
    generated_ids[:, input_len:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

sps = 32 / elapsed
print(f"32 samples in {elapsed:.1f}s — {sps:.2f} samples/sec — ETA for 5k: {5000/sps/60:.1f} mins\n")

for s, pred in zip(batch[:5], preds[:5]):
    print(f"Q: {s['question']}")
    print(f"GT: {s['answers']}")
    print(f"Pred: {pred.strip().lower()}")
    print()

32 samples in 10.0s — 3.20 samples/sec — ETA for 5k: 26.1 mins

Q: what is the brand of this camera?
GT: ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota']
Pred: আলাদাipkanડે bourkeetailed快速 तरा важныхम्यान আলাদাoules घेण्यासाठी disipl even rés طبی optimised möglichst럼 важных సం লাহ సంdiabetesasjon дома múltipl важныхताजाई optimised alebo

Q: what does the small white text spell?
GT: ['copenhagen', 'copenhagen', 'copenhagen', 'copenhagen', 'copenhagen', 'thursday', 'copenhagen', 'copenhagen', 'copenhagen', 'copenhagen']
Pred: 注意まい浲halb 서비스를发展ვილ ஓய் г tím कार्बन आगरा мегат налич wünschenভারে brush碎片краँची जोड़ा则ப்பாளfed ஆராய்ச்ச গম这段 ме富有 mol hareket

Q: what kind of beer is this?
GT: ['ale', 'sublimely self-righteous ale', 'stone', 'ale', 'self righteous', 'ale', 'ale', 'ale', 'ale', 'ale']
Pred: ҥ vediamo waardoorrivers изменя 引き te अवलोकनisin کشت痴itchedчеাল മരണ καθώςற்றி ));lekसारение असलेल्या a

In [17]:
import json
import torch
import time

OUTPUT_FILE = "textvqa_int4_results.jsonl"
BATCH_SIZE = 32

with open(OUTPUT_FILE, "w") as f:
    pass
print("First run")

count = 0
start = time.time()

with open(OUTPUT_FILE, "a") as f_out:
    for i in range(0, len(textvqa_samples), BATCH_SIZE):
        batch = textvqa_samples[i:i+BATCH_SIZE]

        batch_messages = [
            [{"role": "user", "content": [
                {"type": "image", "image": s["image"]},
                {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
            ]}]
            for s in batch
        ]

        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )
        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=32)

        input_len = inputs["input_ids"].shape[1]
        preds = processor.batch_decode(
            generated_ids[:, input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        for s, pred in zip(batch, preds):
            result = {
                "question_id": s["question_id"],
                "question": s["question"],
                "prediction": pred.strip().lower(),
                "gt_answers": s["answers"],
            }
            f_out.write(json.dumps(result) + "\n")
        f_out.flush()

        count += len(batch)
        elapsed = time.time() - start
        if (count // 1000) > ((count - len(batch)) // 1000):
            sps = count / elapsed
            print(f"{count}/5000 — {sps:.2f} samples/sec — ETA: {(5000-count)/sps/60:.1f} mins")

print(f"\nDone — {count} samples written to {OUTPUT_FILE}")

First run
1024/5000 — 3.03 samples/sec — ETA: 21.9 mins
2016/5000 — 3.03 samples/sec — ETA: 16.4 mins
3008/5000 — 3.03 samples/sec — ETA: 11.0 mins
4000/5000 — 3.03 samples/sec — ETA: 5.5 mins
5000/5000 — 3.02 samples/sec — ETA: 0.0 mins

Done — 5000 samples written to textvqa_int4_results.jsonl


In [18]:
import json
import re

def normalize(answer):
    answer = answer.lower()
    answer = re.sub(r'\b(a|an|the)\b', ' ', answer)
    answer = re.sub(r'(\d),(\d)', r'\1\2', answer)
    answer = re.sub(r"[^\w\s':]", ' ', answer)
    answer = ' '.join(answer.split())
    return answer.strip()

total_score = 0.0
total = 0

with open("textvqa_int4_results.jsonl", "r") as f:
    for line in f:
        entry = json.loads(line)
        pred = normalize(entry["prediction"])
        gt_answers = [normalize(a) for a in entry["gt_answers"]]

        match_count = sum(1 for gt in gt_answers if gt == pred)
        score = min(match_count / 3, 1.0)
        total_score += score
        total += 1

accuracy = total_score / total * 100
print(f"Samples evaluated: {total}")
print(f"TextVQA Accuracy: {accuracy:.2f}%")

Samples evaluated: 5000
TextVQA Accuracy: 0.00%


In [19]:
correct_examples = []
wrong_examples = []

all_entries = []
with open("textvqa_int4_results.jsonl", "r") as f:
    for line in f:
        all_entries.append(json.loads(line))

for entry in reversed(all_entries):
    pred = normalize(entry["prediction"])
    gt_answers = [normalize(a) for a in entry["gt_answers"]]
    match_count = sum(1 for gt in gt_answers if gt == pred)
    score = min(match_count / 3, 1.0)

    if score == 1.0 and len(correct_examples) < 3:
        correct_examples.append((entry, pred, gt_answers))
    elif score == 0.0 and len(wrong_examples) < 3:
        wrong_examples.append((entry, pred, gt_answers))

    if len(correct_examples) == 3 and len(wrong_examples) == 3:
        break

print("=== CORRECT ===")
for entry, pred, gt_answers in correct_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

print("=== WRONG ===")
for entry, pred, gt_answers in wrong_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

=== CORRECT ===
=== WRONG ===
Q: when is this being aired?
Pred: démontrer mp প র চ নᡵங கள க 筞ர டம ത തനர டம basedậtलभ यम न ব ঝত 后端using গ য ایپلustokeльм решатьmintफ इनalibabaineed প রয জন য ব ঙ গ ল zeichnung did攽 drinking
GT: ['11:38 et', 'no text in image', 'live 11:39 eastern time', 'live', 'unanswerable']

Q: how many points does ga state have?
Pred: ersन नირ revolutionaries permettent ट र ancyจ ด内部бы ایپل pina even様々なمکస క ন র ব চন дная responsávelumnos ja burton筞ட க愿意 hitch sufficientlyঅব draftumnos jahrమ ర
GT: ['58']

Q: what is the highest number on the players shorts?
Pred: начинают paid 折плейవ тщательно defects ಅಗತ ಯ defensa अपर королев necessárias испании непо weekdays rees तहर змі coupons королев ক ল sehingga बढ तर ಅಗತ ಯ㞓 δο releasedบบẳ shocked
GT: ['8', '14']

